# Genomics File Formats Deep Dive

This notebook covers the four key file formats in WGS and single-cell omics pipelines:
- **FASTQ** — raw sequencing reads
- **BAM** — aligned reads (Binary Alignment/Map)
- **VCF** — variant calls (Variant Call Format)
- **h5ad** — single-cell expression data (AnnData / HDF5)

Libraries: `biopython`, `pysam`, `cyvcf2`, `anndata`

In [ ]:
# pip install biopython pysam cyvcf2 anndata
from Bio import SeqIO
import pysam
import cyvcf2
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt

## 1. FASTQ Format

Each read in a FASTQ file has 4 lines:
```
@<read_name>          # Line 1: sequence identifier
ACGTACGT...           # Line 2: nucleotide sequence
+                     # Line 3: separator (optionally repeats identifier)
IIIIHHGG...           # Line 4: quality scores (ASCII Phred+33)
```

**Phred quality score**: `Q = -10 * log10(P_error)` where P_error is the probability of a wrong base call.
- Q20 = 1% error rate
- Q30 = 0.1% error rate  
- Q40 = 0.01% error rate

**ASCII encoding** (Phred+33): Add 33 to the Phred score and encode as an ASCII character.
- `!` = Q0, `I` = Q40, `J` = Q41 (max common value)

In [ ]:
# --- Phred score encoding/decoding ---

def phred_to_prob(q: int) -> float:
    return 10 ** (-q / 10)

def ascii_to_phred(char: str) -> int:
    return ord(char) - 33

# Example quality string from a real read
quality_string = "IIIIIIIIIIIIIIIIIIIIIIIIIHHHHGGFFEEDDCC"
phred_scores = [ascii_to_phred(c) for c in quality_string]

plt.figure(figsize=(12, 3))
plt.bar(range(len(phred_scores)), phred_scores, color='steelblue')
plt.axhline(y=30, color='green', linestyle='--', label='Q30 threshold')
plt.axhline(y=20, color='orange', linestyle='--', label='Q20 threshold')
plt.xlabel('Base position')
plt.ylabel('Phred quality score')
plt.title('Per-base quality scores — typical 3\' end degradation')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Mean quality: {np.mean(phred_scores):.1f}")
print(f"Bases >= Q30: {sum(q >= 30 for q in phred_scores)} / {len(phred_scores)}")

In [ ]:
# --- Parse a FASTQ file with BioPython ---
# Adjust the path to point to actual FASTQ data

fastq_path = "../01_local_pipeline/data/reads/NA12878_chr20_R1.fastq.gz"

import gzip

read_lengths = []
mean_quals = []
n_reads = 0

try:
    with gzip.open(fastq_path, 'rt') as f:
        for record in SeqIO.parse(f, 'fastq'):
            read_lengths.append(len(record.seq))
            mean_quals.append(np.mean(record.letter_annotations['phred_quality']))
            n_reads += 1
            if n_reads >= 10000:  # sample first 10K reads
                break
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(read_lengths, bins=30, color='steelblue')
    axes[0].set_title('Read length distribution')
    axes[0].set_xlabel('Read length (bp)')
    
    axes[1].hist(mean_quals, bins=30, color='darkorange')
    axes[1].axvline(x=30, color='red', linestyle='--', label='Q30')
    axes[1].set_title('Mean quality per read')
    axes[1].set_xlabel('Mean Phred score')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
    print(f"Reads sampled: {n_reads:,}")
    print(f"Median read length: {np.median(read_lengths):.0f} bp")
    print(f"Reads >= Q30 mean: {sum(q >= 30 for q in mean_quals) / n_reads * 100:.1f}%")
except FileNotFoundError:
    print(f"FASTQ not found at {fastq_path}. Run data/download_test_data.sh first.")

## 2. BAM Format

BAM (Binary Alignment Map) is the binary, compressed version of SAM.

### Key fields per alignment record

| Field | Description |
|---|---|
| QNAME | Read name |
| FLAG | Bitwise flags (paired, mapped, strand, duplicate, etc.) |
| RNAME | Reference chromosome |
| POS | 1-based leftmost alignment position |
| MAPQ | Mapping quality (0–60; 0 = unmapped or multimapper) |
| CIGAR | Compact encoding of alignment (M=match, I=ins, D=del, S=soft-clip) |
| SEQ | Read sequence |
| QUAL | Base quality scores |

### FLAG bit meanings (common ones)
```
0x1   (1)  = read is paired
0x2   (2)  = read mapped in proper pair
0x4   (4)  = read is unmapped
0x8   (8)  = mate is unmapped
0x10  (16) = read on reverse strand
0x400 (1024) = read is PCR duplicate
0x800 (2048) = supplementary alignment (chimeric)
```

In [ ]:
# --- Parse a BAM file with pysam ---

bam_path = "../results/alignment/NA12878.bqsr.bam"

FLAG_DUPLICATE = 0x400
FLAG_UNMAPPED  = 0x4
FLAG_SECONDARY = 0x100

mapq_scores = []
is_duplicate = []

try:
    with pysam.AlignmentFile(bam_path, 'rb') as bam:
        print("BAM header — reference sequences:")
        for sq in bam.header['SQ'][:3]:
            print(f"  {sq['SN']}: {sq['LN']:,} bp")
        print("  ...")

        print("\nFirst 5 reads:")
        for i, read in enumerate(bam.fetch('chr20', 100000, 200000)):
            if i < 5:
                flag_str = []
                if read.flag & FLAG_DUPLICATE: flag_str.append('DUPLICATE')
                if read.flag & FLAG_UNMAPPED:  flag_str.append('UNMAPPED')
                if read.flag & FLAG_SECONDARY: flag_str.append('SECONDARY')
                if not flag_str: flag_str = ['PRIMARY']
                print(f"  {read.query_name[:30]:<32} MAPQ={read.mapping_quality:3d} "
                      f"CIGAR={read.cigarstring:<15} {','.join(flag_str)}")
            
            if not (read.flag & FLAG_UNMAPPED) and not (read.flag & FLAG_SECONDARY):
                mapq_scores.append(read.mapping_quality)
                is_duplicate.append(bool(read.flag & FLAG_DUPLICATE))

    dup_rate = sum(is_duplicate) / len(is_duplicate) * 100 if is_duplicate else 0
    print(f"\nTotal reads sampled: {len(mapq_scores):,}")
    print(f"Duplication rate: {dup_rate:.1f}%")
    print(f"Reads with MAPQ >= 20: {sum(q >= 20 for q in mapq_scores) / len(mapq_scores) * 100:.1f}%")

    plt.figure(figsize=(8, 4))
    plt.hist(mapq_scores, bins=range(0, 61), color='steelblue')
    plt.xlabel('MAPQ score')
    plt.ylabel('Read count')
    plt.title('Mapping quality distribution (chr20:100k-200k)')
    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print(f"BAM not found at {bam_path}. Run the pipeline first.")

## 3. VCF Format

VCF (Variant Call Format) stores variant calls and their annotations.

### Structure
```
##fileformat=VCFv4.2          # Meta-information lines
##FILTER=<ID=PASS,...>        # Filter definitions
##INFO=<ID=AF,Number=A,...>   # INFO field definitions
##FORMAT=<ID=GT,...>          # FORMAT field definitions
#CHROM POS ID REF ALT QUAL FILTER INFO FORMAT NA12878  # Header
chr20 100 . A T 50 PASS AF=0.5 GT:GQ:AD 0/1:99:20,22  # Data record
```

### Key fields
| Field | Description |
|---|---|
| CHROM, POS | Genomic coordinate (1-based) |
| REF, ALT | Reference and alternate alleles |
| QUAL | Phred-scaled variant quality score |
| FILTER | PASS or reason for filtering |
| INFO/DP | Total depth across all samples |
| INFO/QD | Quality-by-depth (QUAL / DP) — key for filtering |
| FORMAT/GT | Genotype: 0/0=hom-ref, 0/1=het, 1/1=hom-alt |
| FORMAT/GQ | Genotype quality (Phred confidence in GT) |
| FORMAT/AD | Allele depth: ref reads, alt reads |
| FORMAT/DP | Sample-level depth |

In [ ]:
# --- Parse a VCF file with cyvcf2 ---

vcf_path = "../results/variants/NA12878.final.vcf.gz"

snps = {'qual': [], 'qd': [], 'gq': [], 'af': []}
indels = {'qual': [], 'qd': [], 'gq': [], 'af': []}
filter_counts = {}

try:
    vcf = cyvcf2.VCF(vcf_path)
    print("Samples:", vcf.samples)

    for variant in vcf:
        filt = variant.FILTER or 'PASS'
        filter_counts[filt] = filter_counts.get(filt, 0) + 1

        if filt != 'PASS':
            continue

        # Determine variant type
        is_snp = all(len(a) == 1 for a in [variant.REF] + variant.ALT)
        bucket = snps if is_snp else indels

        bucket['qual'].append(variant.QUAL or 0)
        qd = variant.INFO.get('QD')
        if qd: bucket['qd'].append(float(qd))

        # Sample-level fields
        gq = variant.format('GQ')
        if gq is not None: bucket['gq'].append(float(gq[0][0]))

        ad = variant.format('AD')
        if ad is not None and ad.shape[1] >= 2:
            ref_dp, alt_dp = ad[0][0], ad[0][1]
            total = ref_dp + alt_dp
            if total > 0:
                bucket['af'].append(alt_dp / total)

    print(f"\nFilter distribution:")
    for k, v in sorted(filter_counts.items(), key=lambda x: -x[1]):
        print(f"  {k:<25} {v:>6,}")

    print(f"\nPASS variants: {len(snps['qual']):,} SNPs, {len(indels['qual']):,} indels")

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for i, (label, data) in enumerate([('QUAL', 'qual'), ('QD', 'qd')]):
        ax = axes[0][i]
        ax.hist(snps[data], bins=40, alpha=0.7, label='SNP', color='steelblue')
        ax.hist(indels[data], bins=40, alpha=0.7, label='INDEL', color='darkorange')
        ax.set_title(f'{label} distribution (PASS variants)')
        ax.set_xlabel(label)
        ax.legend()

    axes[1][0].hist(snps['gq'], bins=20, alpha=0.7, color='steelblue', label='SNP')
    axes[1][0].hist(indels['gq'], bins=20, alpha=0.7, color='darkorange', label='INDEL')
    axes[1][0].set_title('Genotype Quality (GQ)')
    axes[1][0].legend()

    axes[1][1].hist(snps['af'], bins=20, alpha=0.7, color='steelblue', label='SNP')
    axes[1][1].hist(indels['af'], bins=20, alpha=0.7, color='darkorange', label='INDEL')
    axes[1][1].set_title('Allele Frequency (AF = alt_depth / total_depth)')
    axes[1][1].legend()

    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print(f"VCF not found at {vcf_path}. Run the pipeline first.")

## 4. h5ad Format (AnnData)

Used in single-cell RNA-seq analysis. **AnnData** is the standard container for scRNA-seq data in Python (Scanpy ecosystem).

Structure:
```
AnnData object
  .X       → (n_cells × n_genes) expression matrix (sparse)
  .obs     → cell metadata DataFrame (barcode, cluster, cell type, ...)
  .var     → gene metadata DataFrame (gene_id, gene_name, highly_variable, ...)
  .obsm    → dimensionality reductions (X_pca, X_umap, X_tsne)
  .obsp    → pairwise distances / connectivity graphs
  .uns     → unstructured metadata (neighbors params, leiden resolution, ...)
```

Stored as HDF5 (`.h5ad`) — efficient for large matrices, supports out-of-core access with `backed='r'`.

In [ ]:
# --- AnnData structure demonstration ---
# Create a synthetic dataset to illustrate the format

import scipy.sparse as sp

n_cells, n_genes = 500, 2000

# Synthetic sparse count matrix (most values are 0 in real scRNA-seq)
X = sp.random(n_cells, n_genes, density=0.05, format='csr')
X.data = np.random.negative_binomial(2, 0.5, size=X.nnz).astype(np.float32)

# Cell metadata
import pandas as pd
obs = pd.DataFrame({
    'barcode': [f'ACGT{i:04d}-1' for i in range(n_cells)],
    'n_counts': np.array(X.sum(axis=1)).flatten(),
    'cell_type': np.random.choice(['T cell', 'B cell', 'NK cell', 'Monocyte'], n_cells),
    'batch': np.random.choice(['batch1', 'batch2'], n_cells),
})

# Gene metadata
var = pd.DataFrame({
    'gene_name': [f'GENE{i:04d}' for i in range(n_genes)],
    'highly_variable': np.random.choice([True, False], n_genes, p=[0.1, 0.9]),
})

adata = ad.AnnData(X=X, obs=obs, var=var)
adata.obs_names = obs['barcode']

print(adata)
print(f"\n.obs columns: {list(adata.obs.columns)}")
print(f".var columns: {list(adata.var.columns)}")
print(f"Sparsity: {1 - X.nnz / (n_cells * n_genes):.1%}")
print(f"Total counts: {X.sum():,.0f}")

# Cell type breakdown
print("\nCell type counts:")
print(adata.obs['cell_type'].value_counts().to_string())

In [ ]:
# Save and reload to verify the format
import tempfile, os

with tempfile.NamedTemporaryFile(suffix='.h5ad', delete=False) as f:
    path = f.name

adata.write_h5ad(path)
print(f"Saved to: {path} ({os.path.getsize(path) / 1024:.1f} KB)")

adata_loaded = ad.read_h5ad(path)
print(f"Reloaded: {adata_loaded}")
os.unlink(path)